# Analysis: Mechanistic Variability

We have trained several different Tiny RNN model architectures on a simple reversal task.
What we found is that GRU's trained on the task have varying underlying gating mechanisms.
We sought to minimise this variability by simplifying the architecture, while retaining the performance.


In [2]:
## setup ##
import os
os.chdir('/ceph/behrens/wsilver/reversal/code/')

from NM_TinyRNN.code.measures import analysis
from NM_TinyRNN.code.measures import mechanistic_variability as mech_var
from NM_TinyRNN.code.measures import plotting_stats
from NM_TinyRNN.code.measures import performance as perf

from NM_TinyRNN.code.models import submit_jobs # this is the code used to train models!

import numpy as np
import pandas as pd
import torch #for testing a few things
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from importlib import reload

CODE_DIR = Path('.') ## OBS THIS MAY NEED TO BE ADJUSTED!
SAVE_PATH = CODE_DIR/'NM_TinyRNN/data/rnns'
DATA_PATH = Path('./NM_TinyRNN/data/AB_behaviour/')

%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# load data
#construct analysis_df and have a look at a subset of data!
info_df = submit_jobs.get_DA_info_df() #code used to submit jobs
all_models_df = analysis.get_analysis_df(info_df, mode='all') #code to aggregate all paths and some data
performance_df = perf.get_performance_df(all_models_df) #code to recompute evaluations and aggregate
best_model_df = perf.select_best_outer(performance_df) #

print("Available model types (model_type2):")
print(all_models_df.model_type2.unique())

Loaded analysis paths from cache: NM_TinyRNN/data/analysis/analysis_df.htsv


KeyboardInterrupt: 

## Similarities in activations (within model architecture)

In [ ]:
# compute similarities

#construct analysis_df and have a look at a subset of data!
info_df = submit_jobs.get_DA_info_df() #code used to submit jobs
all_models_df = analysis.get_analysis_df(info_df, mode='all') #code to aggregate all paths and some data
performance_df = perf.get_performance_df(all_models_df) #code to recompute evaluations and aggregate
best_model_df = perf.select_best_outer(performance_df) #
subject_level_df = perf.compute_outer_mean(best_model_df)

print("Available model types (model_type2):")
print(all_models_df.model_type2.unique())

print("\nAvailable columns:")
print([col for col in all_models_df.columns if 'CE' in col or 'type' in col])


## Similarities in weights (within model architecture)

In [ ]:
# Across all gating architectures - gating variability.
## Figure 2: variability in GRU model mechanisms.
model_list = ['GRU','GRU+BC','lightGRU+BC','monoGRU+BC','monoGRU+BC-DB']
USE_BEST_MODELS = False

source_df = best_model_df if USE_BEST_MODELS else all_models_df
select_models = source_df.query('hidden_size==2 and input_encoding == "unipolar"').copy()
select_models = select_models[select_models.model_type2.isin(model_list)]
select_models['model_id'] = select_models.model_type2
sim_df = mech_var.compute_similarities(select_models)